# FDTD field visualization

Loads custom binary output from `plt/` (written by `writeNumpyOutput` in `fdtd.cpp`).

Run the simulation first:
```bash
make -j && ./main3d.gnu.ex inputs
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

from IPython.display import Video, display

OUTPUT_DIR = Path("plt")
COMPONENTS = ["Ex", "Ey", "Ez", "Bx", "By", "Bz"]

In [ ]:
def list_steps(output_dir=OUTPUT_DIR):
    """Return sorted list of (step, meta_path) pairs."""
    metas = sorted(output_dir.glob("step_*_meta.json"))
    steps = []
    for meta_path in metas:
        meta = json.loads(meta_path.read_text())
        steps.append((meta["step"], meta_path))
    return steps


def load_step(meta_path):
    """Load one dump: returns (fields, meta). fields shape is (nx, ny, nz, 6)."""
    meta = json.loads(Path(meta_path).read_text())
    bin_path = Path(meta_path).with_name(Path(meta_path).name.replace("_meta.json", "_fields.bin"))
    fields = np.fromfile(bin_path, dtype=np.float64).reshape(meta["shape"])
    return fields, meta


def load_series(output_dir=OUTPUT_DIR):
    """Load all dumps as a list of (step, time, fields, meta)."""
    series = []
    for step, meta_path in list_steps(output_dir):
        fields, meta = load_step(meta_path)
        series.append((meta["step"], meta["time"], fields, meta))
    return series


def coord_axes(meta):
    """Cell-centered 1D coordinate arrays for x, y, z."""
    nx, ny, nz, _ = meta["shape"]
    prob_lo = np.array(meta["prob_lo"])
    prob_hi = np.array(meta["prob_hi"])
    x = np.linspace(prob_lo[0], prob_hi[0], nx, endpoint=False) + 0.5 * (prob_hi[0] - prob_lo[0]) / nx
    y = np.linspace(prob_lo[1], prob_hi[1], ny, endpoint=False) + 0.5 * (prob_hi[1] - prob_lo[1]) / ny
    z = np.linspace(prob_lo[2], prob_hi[2], nz, endpoint=False) + 0.5 * (prob_hi[2] - prob_lo[2]) / nz
    return x, y, z


# Sinwave IC settings (keep in sync with `inputs`)
SINWAVE_AMPLITUDE = 0.01
SINWAVE_WAVELENGTH = -1  # <0 => one wavelength along propagation direction
SINWAVE_DIR = 2          # 0=x, 1=y, 2=z
SINWAVE_POL = 0          # 0=Ex, 1=Ey, 2=Ez


def expected_sinwave(coord, axis, meta, comp):
    """Analytic cell-centered sinwave for validating step 0 output."""
    axis_idx = {"x": 0, "y": 1, "z": 2}[axis]
    comp_idx = COMPONENTS.index(comp)
    if comp_idx != SINWAVE_POL or axis_idx != SINWAVE_DIR:
        return None
    prob_lo = np.array(meta["prob_lo"])
    prob_hi = np.array(meta["prob_hi"])
    domain_len = prob_hi[SINWAVE_DIR] - prob_lo[SINWAVE_DIR]
    wavelength = domain_len if SINWAVE_WAVELENGTH < 0 else SINWAVE_WAVELENGTH
    kw = 2 * np.pi / wavelength
    return SINWAVE_AMPLITUDE * np.sin(kw * coord)


steps = list_steps()
if not steps:
    raise FileNotFoundError(f"No output in {OUTPUT_DIR.resolve()}. Run ./main3d.gnu.ex inputs first.")

print(f"Found {len(steps)} dumps: steps {[s for s, _ in steps]}")

## 2D mid-plane slices (all six components)

In [ ]:
STEP_TO_PLOT = steps[-1][0]  # latest step; change index or set an integer step
meta_path = next(p for s, p in steps if s == STEP_TO_PLOT)
fields, meta = load_step(meta_path)
x, y, z = coord_axes(meta)

ix, iy, iz = len(x) // 2, len(y) // 2, len(z) // 2
SLICE_AXIS = "z"  # "x", "y", or "z"
SLICE_INDEX = iz

if SLICE_AXIS == "z":
    slices = [fields[:, :, SLICE_INDEX, c] for c in range(6)]
    extent = [x[0], x[-1], y[0], y[-1]]
    xlabel, ylabel = "x", "y"
    plane_label = f"z = {z[SLICE_INDEX]:.4g}"
elif SLICE_AXIS == "y":
    slices = [fields[:, SLICE_INDEX, :, c] for c in range(6)]
    extent = [x[0], x[-1], z[0], z[-1]]
    xlabel, ylabel = "x", "z"
    plane_label = f"y = {y[SLICE_INDEX]:.4g}"
else:
    slices = [fields[SLICE_INDEX, :, :, c] for c in range(6)]
    extent = [y[0], y[-1], z[0], z[-1]]
    xlabel, ylabel = "y", "z"
    plane_label = f"x = {x[SLICE_INDEX]:.4g}"

vmax = max(abs(s).max() for s in slices) or 1.0
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
fig.suptitle(f"step {meta['step']}  t = {meta['time']:.3e} s  ({plane_label})")

for ax, comp, slc in zip(axes.ravel(), COMPONENTS, slices):
    im = ax.imshow(slc.T, origin="lower", extent=extent, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_title(comp)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.show()

## 1D line cut through domain center (animation)

Line cut at the domain center along `LINE_AXIS`. Saves an MP4 (requires [ffmpeg](https://ffmpeg.org/) on your PATH). Useful for watching sin-wave propagation in +z.

In [ ]:
LINE_AXIS = "z"  # axis to plot along: "x", "y", or "z"
COMP = "Ex"      # component name from COMPONENTS
comp_idx = COMPONENTS.index(COMP)

OUTPUT_MP4 = Path("plt") / f"{COMP.lower()}_line_{LINE_AXIS}.mp4"
FPS = 5

series = load_series()
meta0 = series[0][3]
x, y, z = coord_axes(meta0)
ix, iy, iz = len(x) // 2, len(y) // 2, len(z) // 2

if LINE_AXIS == "z":
    coord = z
    lines = [f[ix, iy, :, comp_idx] for _, _, f, _ in series]
elif LINE_AXIS == "y":
    coord = y
    lines = [f[ix, :, iz, comp_idx] for _, _, f, _ in series]
else:
    coord = x
    lines = [f[:, iy, iz, comp_idx] for _, _, f, _ in series]

ymin = min(line.min() for line in lines)
ymax = max(line.max() for line in lines)
pad = 0.05 * max(abs(ymin), abs(ymax), 1e-12)
ylim = (ymin - pad, ymax + pad)

expected = expected_sinwave(coord, LINE_AXIS, meta0, COMP)
if expected is not None:
    rms0 = float(np.sqrt(np.mean((lines[0] - expected) ** 2)))
    print(f"step 0 RMS error vs analytic sin: {rms0:.2e}")

fig, ax = plt.subplots(figsize=(8, 4))
(line_plot,) = ax.plot(coord, lines[0], "-", lw=1.5, label="simulation")
if expected is not None:
    ax.plot(coord, expected, "--", color="gray", lw=1.0, alpha=0.8, label="analytic t=0")
    ax.legend(loc="upper right")
ax.set_xlim(coord[0], coord[-1])
ax.set_ylim(ylim)
ax.set_xlabel(LINE_AXIS)
ax.set_ylabel(COMP)
ax.grid(True, alpha=0.3)
title = ax.set_title("")


def update(i):
    step, time, _, _ = series[i]
    line_plot.set_ydata(lines[i])
    title.set_text(f"{COMP} along {LINE_AXIS} at center  step {step}  t = {time:.3e} s")
    return line_plot, title


anim = animation.FuncAnimation(fig, update, frames=len(series), interval=1000 // FPS, blit=False)
writer = animation.FFMpegWriter(fps=FPS, metadata={"title": f"{COMP} line cut"})
anim.save(OUTPUT_MP4, writer=writer)
plt.close(fig)

display(Video(str(OUTPUT_MP4.resolve())))
print(f"Saved {OUTPUT_MP4.resolve()}")

## Animation over time (single component, z mid-plane)

Re-run the simulation with `fdtd.plot_int > 0` to produce multiple dumps. Saves an MP4 (requires [ffmpeg](https://ffmpeg.org/) on your PATH).

In [ ]:
series = load_series()
ANIM_COMP = "Ex"
comp_idx = COMPONENTS.index(ANIM_COMP)
iz = series[0][2].shape[2] // 2

frames = [f[:, :, iz, comp_idx] for _, _, f, _ in series]
vmax = max(abs(frame).max() for frame in frames) or 1.0
x, y, _ = coord_axes(series[0][3])
extent = [x[0], x[-1], y[0], y[-1]]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(frames[0].T, origin="lower", extent=extent, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xlabel("x")
ax.set_ylabel("y")
title = ax.set_title("")
fig.colorbar(im, ax=ax, label=ANIM_COMP)


def update(i):
    step, time, _, _ = series[i]
    im.set_data(frames[i].T)
    title.set_text(f"{ANIM_COMP}  step {step}  t = {time:.3e} s  (z mid-plane)")
    return [im, title]


OUTPUT_MP4 = Path("plt") / f"{ANIM_COMP.lower()}_zmidplane.mp4"
FPS = 5

anim = animation.FuncAnimation(fig, update, frames=len(series), interval=1000 // FPS, blit=False)
writer = animation.FFMpegWriter(fps=FPS, metadata={"title": ANIM_COMP})
anim.save(OUTPUT_MP4, writer=writer)
plt.close(fig)

display(Video(str(OUTPUT_MP4.resolve())))
print(f"Saved {OUTPUT_MP4.resolve()}")